# Objectifs d'apprentissage – Semaine 5

À l'issue de cette semaine, l'étudiant sera capable de :

- **Définir** la transformée en Z bilatérale d'une séquence.
- **Déterminer** la région de convergence (ROC) pour différentes séquences (causales, anticausales, bilatérales).
- **Calculer** la transformée en Z de séquences usuelles (impulsion, échelon, exponentielle, sinusoïde).
- **Représenter** les pôles et zéros d'une fonction rationnelle.
- **Analyser** la stabilité d'un système LTI à partir de la position des pôles par rapport au cercle unité.
- **Utiliser** Python (numpy, scipy.signal, matplotlib) pour calculer et visualiser les pôles et zéros.
- **Appliquer** la transformée en Z à la résolution d'équations aux différences.

# 1. Rappels théoriques

## 1.1 Définition de la transformée en Z
La transformée en Z bilatérale d'une séquence $x[n]$ est définie par :
$$ X(z) = \sum_{n=-\infty}^{\infty} x[n] z^{-n} $$
où $z$ est une variable complexe.

## 1.2 Région de convergence (ROC)
La ROC est l'ensemble des valeurs de $z$ pour lesquelles la série converge. Elle dépend de la nature de la séquence :
- **Séquence causale** ($x[n]=0$ pour $n<0$) : ROC est l'extérieur d'un cercle de rayon $r$ (soit $|z| > r$).
- **Séquence anticausale** ($x[n]=0$ pour $n>0$) : ROC est l'intérieur d'un cercle (soit $|z| < r$).
- **Séquence bilatérale** : ROC est un anneau $r_1 < |z| < r_2$.

La ROC ne contient aucun pôle.

## 1.3 Pôles et zéros
Si $X(z)$ est une fonction rationnelle, elle s'écrit :
$$ X(z) = \frac{N(z)}{D(z)} $$
Les zéros sont les racines de $N(z)$, les pôles sont les racines de $D(z)$.

## 1.4 Stabilité
Un système LTI est stable (BIBO) si et seulement si sa ROC contient le cercle unité ($|z|=1$).
Pour un système causal, cela équivaut à avoir tous les pôles à l'intérieur du cercle unité ($|p_i| < 1$).

---

# 2. Démonstrations Python

## 2.1 Transformée en Z de séquences usuelles

Nous allons calculer symboliquement les transformées en Z de quelques séquences simples à l'aide de `sympy`, puis visualiser les pôles et zéros avec `scipy.signal`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import sympy as sp

sp.init_printing()

print("Bibliothèques chargées.")

In [ ]:
# Définition des symboles
z = sp.Symbol('z', complex=True)
n = sp.Symbol('n', integer=True)
a = sp.Symbol('a', real=True)

# 1. Impulsion unité
x1 = sp.KroneckerDelta(n, 0)
X1 = sp.summation(x1 * z**(-n), (n, -sp.oo, sp.oo))
print("Transformée de l'impulsion :", X1)

# 2. Échelon unité (causal)
X2 = sp.summation(z**(-n), (n, 0, sp.oo))
print("Transformée de l'échelon causal :", X2)

# 3. Exponentielle causale a^n u[n]
X3 = sp.summation((a*z**(-1))**n, (n, 0, sp.oo))
print("Transformée de a^n u[n] :", X3)

## 2.2 Tracé des pôles et zéros

Nous allons utiliser `scipy.signal.tf2zpk` pour obtenir les pôles et zéros d'une fonction de transfert et les tracer dans le plan complexe.

In [ ]:
def plot_pole_zero(b, a, title='Diagramme pôles-zéros'):
    """Trace le diagramme pôles-zéros à partir des coefficients du numérateur (b) et du dénominateur (a)."""
    zeros, poles, gain = signal.tf2zpk(b, a)
    
    # Cercle unité
    theta = np.linspace(0, 2*np.pi, 200)
    plt.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5, label='Cercle unité')
    
    # Pôles et zéros
    plt.scatter(np.real(zeros), np.imag(zeros), marker='o', s=100, facecolors='none', edgecolors='b', label='Zéros')
    plt.scatter(np.real(poles), np.imag(poles), marker='x', s=100, color='r', label='Pôles')
    
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.grid(True, alpha=0.3)
    plt.axis('equal')
    plt.xlim([-2, 2])
    plt.ylim([-2, 2])
    plt.xlabel('Partie réelle')
    plt.ylabel('Partie imaginaire')
    plt.title(title)
    plt.legend()
    plt.show()
    
    return zeros, poles

# Exemple : H(z) = 1 / (1 - 0.5 z^{-1})  -> pôle en z=0.5
b = [1]
a = [1, -0.5]
plot_pole_zero(b, a, 'Pôle en 0.5 (stable)')

# Exemple instable : pôle en 1.2
b2 = [1]
a2 = [1, -1.2]
plot_pole_zero(b2, a2, 'Pôle en 1.2 (instable)')

# Exemple avec zéros
b3 = [1, -0.5]  # zéro en 0.5
a3 = [1, -1.2]  # pôle en 1.2
plot_pole_zero(b3, a3, 'Zéro en 0.5, pôle en 1.2')

## 2.3 Stabilité et ROC

Pour un système causal, la stabilité est équivalente à tous les pôles à l'intérieur du cercle unité. Vérifions avec des exemples.

In [ ]:
def check_stability(b, a):
    """Vérifie si un système causal est stable (pôles dans le cercle unité)."""
    zeros, poles, gain = signal.tf2zpk(b, a)
    stable = all(np.abs(p) < 1 for p in poles)
    print(f"Pôles : {poles}")
    print(f"Stable (causal) : {stable}")
    return stable

# Système stable
b = [1]
a = [1, -0.5]
check_stability(b, a)

# Système instable
b = [1]
a = [1, -1.2]
check_stability(b, a)

# 3. Exercices – Semaine 5

## Exercice 1 – Transformée en Z et ROC

Déterminer la transformée en Z et la ROC pour les séquences suivantes :

1. $x[n] = (0.5)^n u[n]$
2. $x[n] = - (0.5)^n u[-n-1]$
3. $x[n] = (0.5)^n u[n] + (2)^n u[-n-1]$
4. $x[n] = \delta[n] + \delta[n-1]$

**Correction** (à compléter) :

1. $X(z) = \frac{1}{1 - 0.5 z^{-1}}$, ROC : $|z| > 0.5$.
2. $X(z) = \frac{1}{1 - 0.5 z^{-1}}$, ROC : $|z| < 0.5$ (car séquence anticausale).
3. $X(z) = \frac{1}{1 - 0.5 z^{-1}} + \frac{1}{1 - 2 z^{-1}}$ (à vérifier, attention aux signes), ROC : $0.5 < |z| < 2$.
4. $X(z) = 1 + z^{-1}$, ROC : tout le plan complexe sauf $z=0$ (séquence finie).

## Exercice 2 – Pôles, zéros et stabilité

Considérons un système LTI causal de fonction de transfert :
$$ H(z) = \frac{1 - 0.5 z^{-1}}{1 - 1.5 z^{-1} + 0.5 z^{-2}} $$

1. Déterminer les pôles et les zéros.
2. Tracer le diagramme pôles-zéros.
3. Le système est-il stable ? Justifier.
4. Déterminer la réponse impulsionnelle $h[n]$ (décomposition en éléments simples).

**Code** :

In [ ]:
b = [1, -0.5]
a = [1, -1.5, 0.5]

# Pôles et zéros
zeros, poles, gain = signal.tf2zpk(b, a)
print(f"Zéros : {zeros}")
print(f"Pôles : {poles}")

# Tracé
plot_pole_zero(b, a, 'H(z)')

# Stabilité
stable = all(np.abs(p) < 1 for p in poles)
print(f"Stable : {stable}")

# Décomposition en éléments simples
# On peut utiliser scipy.signal.residuez
r, p, k = signal.residuez(b, a)
print(f"Résidus : {r}")
print(f"Pôles : {p}")
print(f"Terme direct : {k}")
# h[n] = r1 * p1^n + r2 * p2^n pour n>=0

## Exercice 3 – Équation aux différences

Un système LTI causal est décrit par :
$$ y[n] - 0.6 y[n-1] + 0.08 y[n-2] = x[n] - 0.2 x[n-1] $$

1. Déterminer la fonction de transfert $H(z)$.
2. Trouver les pôles et zéros.
3. Le système est-il stable ?
4. Déterminer la réponse impulsionnelle $h[n]$.
5. Calculer la réponse à un échelon unité $x[n] = u[n]$.

**Correction** (à compléter) :

In [ ]:
b = [1, -0.2]
a = [1, -0.6, 0.08]

# Pôles et zéros
zeros, poles, gain = signal.tf2zpk(b, a)
print("Zéros :", zeros)
print("Pôles :", poles)
print("Stable :", all(np.abs(p) < 1 for p in poles))

# Réponse impulsionnelle
r, p, k = signal.residuez(b, a)
print("Résidus :", r)
print("Pôles :", p)
print("h[n] = ", r[0], "*", p[0], "^n +", r[1], "*", p[1], "^n")

# Réponse à un échelon
# On peut simuler avec scipy.signal.lfilter
N = 20
n = np.arange(N)
x = np.ones(N)
y = signal.lfilter(b, a, x)
plt.stem(n, y, basefmt=' ')
plt.xlabel('n')
plt.ylabel('y[n]')
plt.title('Réponse à un échelon')
plt.grid()
plt.show()

## Exercice 4 – Stabilité et ROC

Pour les fonctions de transfert suivantes, déterminer toutes les ROC possibles et indiquer pour chacune si le système est stable et/ou causal.

1. $H(z) = \frac{1}{1 - 0.5 z^{-1}}$
2. $H(z) = \frac{1}{(1 - 0.5 z^{-1})(1 - 2 z^{-1})}$
3. $H(z) = \frac{1 - 0.5 z^{-1}}{1 - 1.5 z^{-1} + 0.5 z^{-2}}$

**Réponses** :

1. ROC possible : $|z|>0.5$ (causal, stable), $|z|<0.5$ (anticausal, instable car la ROC ne contient pas le cercle unité).
2. Pôles en 0.5 et 2. ROC possibles : $|z|>2$ (causal, instable car pôle en 2), $0.5<|z|<2$ (bilatéral, stable car cercle unité inclus), $|z|<0.5$ (anticausal, instable).
3. Pôles en 0.5 et 1. ROC possible : $|z|>1$ (causal, instable car pôle sur le cercle), $0.5<|z|<1$ (bilatéral, instable car cercle non inclus), $|z|<0.5$ (anticausal, instable).

## Exercice 5 – Transformée en Z inverse (niveau avancé)

Soit $X(z) = \frac{1}{1 - z^{-1} + 0.25 z^{-2}}$.

1. Déterminer les pôles.
2. Pour chaque ROC possible, déterminer la séquence $x[n]$ correspondante.
3. Vérifier numériquement pour la ROC causale (i.e., $|z| > |p_{max}|$) en générant les premiers termes de $x[n]$ et en comparant avec `scipy.signal.lfilter`.

**Correction** :

Pôles : $z = 0.5$ (double).
ROC causale : $|z|>0.5$ -> $x[n] = (n+1)(0.5)^n u[n]$.
ROC anticausale : $|z|<0.5$ -> $x[n] = -(n+1)(0.5)^n u[-n-1]$.
Pas de ROC bilatérale (pôles multiples).

Vérification numérique :

In [ ]:
b = [1]
a = [1, -1, 0.25]  # 1 - z^{-1} + 0.25 z^{-2}

# Réponse impulsionnelle pour ROC causale
N = 10
impulse = np.zeros(N)
impulse[0] = 1
h = signal.lfilter(b, a, impulse)

print("h[n] calculé :", h)
print("Formule : (n+1)*(0.5)^n")
n = np.arange(N)
h_theo = (n+1) * (0.5)**n
print("h_theo :", h_theo)
print("Erreur max :", np.max(np.abs(h - h_theo)))

# 4. Livrable Semaine 5

- Remplir les cellules de code avec vos solutions.
- Répondre aux questions théoriques dans les cellules markdown.
- Inclure des commentaires explicatifs.
- Rendre le notebook complet (cellules exécutées).

**Date de rendu** : à définir par l'enseignant.

---